## 과제 1

### LSTM 셀 구현

In [ ]:
#  LSTMCell

In [4]:
import torch
import torch.nn as nn
import torchvision.transforms as transforms
import torchvision.datasets as dataset
from torch.autograd import Variable
from torch.nn import Parameter
from torch import Tensor
import torch.nn.functional as F
from torch.utils.data import DataLoader
import math

device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
cuda = True if torch.cuda.is_available() else False
print(cuda)
Tensor = torch.cuda.FloatTensor if cuda else torch.FloatTensor

torch.manual_seed(125)  # 랜덤 시드 설정
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(125)

True


In [5]:
import torchvision.transforms as transforms   # 데이터 정규화

mnist_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (1.0,))
])

In [6]:
from torchvision.datasets import MNIST

download_root = 'MNIST_DATASET/'

train_dataset = MNIST(download_root, transform=mnist_transform, train=True, download=True)
valid_dataset = MNIST(download_root, transform=mnist_transform, train=False, download=True)
test_dataset = MNIST(download_root, transform=mnist_transform, train=False, download=True)

In [7]:
batch_size = 64
train_loader = DataLoader(dataset=train_dataset,
                         batch_size=batch_size,
                         shuffle=True)  # 배치 섞음
valid_loader = DataLoader(dataset=test_dataset,
                         batch_size=batch_size,
                         shuffle=True)
test_loader = DataLoader(dataset=test_dataset,
                         batch_size=batch_size,
                         shuffle=True)

In [8]:
batch_size = 100
n_iters = 6000  # 총 미니배치 횟수
num_epochs = n_iters / (len(train_dataset) / batch_size)
num_epochs = int(num_epochs)

In [9]:
class LSTMCell(nn.Module):
    def __init__(self, input_size, hidden_size, bias=True):
        super(LSTMCell, self).__init__()
        self.input_size = input_size
        self.hidden_size = hidden_size
        self.bias = bias
        self.x2h = nn.Linear(input_size, 4 * hidden_size, bias=bias)
        # 4개의 게이트(ingate, forgetgate, cellgate, outgate)를 한 번에 계산하기 위해 출력 크기 = 4 * hidden_size
        # 현재 입력 변환
        self.h2h = nn.Linear(hidden_size, 4 * hidden_size, bias=bias)
        # 이전 은닉 상태 변환

        self.reset_parameters()

    def reset_parameters(self):  # LSTMCell 안의 모든 가중치와 bias 초기화 (기울기 폭발/소실 최소화)
        std = 1.0 / math.sqrt(self.hidden_size)
        for w in self.parameters():
            w.data.uniform_(-std, std)

    def forward(self, x, hidden):
        hx, cx = hidden  # 이전 시점의 은닉, 셀 상태 꺼냄
        x = x.view(-1, x.size(1))

        gates = self.x2h(x) + self.h2h(hx)  # 게이트 선형결합  x2h = W_x*x_t+b_x / h2h = W_h*h_t-1+b_h
        gates = gates.squeeze()
        ingate, forgetgate, cellgate, outgate = gates.chunk(4, 1)

        ingate = F.sigmoid(ingate)  # 입력게이트 -> 시그모이드
        forgetgate = F.sigmoid(forgetgate)  # 망각게이트 -> 시그모이드
        cellgate = F.tanh(cellgate)  # 셀 스테이트 -> tanh
        outgate = F.sigmoid(outgate)  # 출력게이트 -> 시그모이드

        # Hadamard고 product
        cy = torch.mul(cx, forgetgate) +  torch.mul(ingate, cellgate)  # 새 셀 상태 c_t 계산
        hy = torch.mul(outgate, F.tanh(cy))  # 새 은닉 상태 h_t 계산
        return (hy, cy)

In [10]:
class LSTMModel(nn.Module):
    def __init__(self, input_dim, hidden_dim, layer_dim, output_dim, bias=True):
        super(LSTMModel, self).__init__()
        self.hidden_dim = hidden_dim

        self.layer_dim = layer_dim
        self.lstm = LSTMCell(input_dim, hidden_dim, layer_dim)
        self.fc = nn.Linear(hidden_dim, output_dim)

    def forward(self, x):
        if torch.cuda.is_available():
            h0 = Variable(torch.zeros(self.layer_dim, x.size(0), self.hidden_dim).cuda())
        else:
            h0 = Variable(torch.zeros(self.layer_dim, x.size(0), self.hidden_dim))

        if torch.cuda.is_available():
            c0 = Variable(torch.zeros(self.layer_dim, x.size(0), self.hidden_dim).cuda())
        else:
            c0 = Variable(torch.zeros(self.layer_dim, x.size(0), hidden_dim))

        outs = []
        cn = c0[0,:,:]
        hn = h0[0,:,:]

        for seq in range(x.size(1)):
            hn, cn = self.lstm(x[:,seq,:], (hn,cn))
            outs.append(hn)

        out = outs[-1].squeeze()
        out = self.fc(out)
        return out

In [11]:
input_dim = 28
hidden_dim = 128
layer_dim = 1
output_dim = 10

model = LSTMModel(input_dim, hidden_dim, layer_dim, output_dim)
if torch.cuda.is_available():
    model.cuda()
criterion = nn.CrossEntropyLoss()
learning_rate = 0.1
optimizer = torch.optim.SGD(model.parameters(), lr=learning_rate)

In [12]:
seq_dim = 28
loss_list = []
iter = 0
for epoch in range(num_epochs):
    for i, (images, labels) in enumerate(train_loader):
        if torch.cuda.is_available():
            images = Variable(images.view(-1, seq_dim, input_dim).cuda())
            labels = Variable(labels.cuda())
        else:
            images = Variable(images.view(-1, seq_dim, input_dim))
            labels = Variable(labels)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)

        if torch.cuda.is_available():
            loss.cuda()

        loss.backward()
        optimizer.step()
        loss_list.append(loss.item())
        iter += 1

        if iter % 500 == 0:
            correct = 0
            total = 0
            for images, labels in valid_loader:
                if torch.cuda.is_available():
                    images = Variable(images.view(-1, seq_dim, input_dim).cuda())
                else:
                    images = Variable(images.view(-1 , seq_dim, input_dim))

                outputs = model(images)
                _, predicted = torch.max(outputs.data, 1)

                total += labels.size(0)
                if torch.cuda.is_available():
                    correct += (predicted.cpu() == labels.cpu()).sum()
                else:
                    correct += (predicted == labels).sum()

            accuracy = 100 * correct / total
            print('Iteration: {}. Loss: {}. Accuracy: {}'.format(iter, loss.item(), accuracy))

Iteration: 500. Loss: 2.237457275390625. Accuracy: 21.420000076293945
Iteration: 1000. Loss: 0.9968994855880737. Accuracy: 74.94999694824219
Iteration: 1500. Loss: 0.42673957347869873. Accuracy: 88.13999938964844
Iteration: 2000. Loss: 0.30981600284576416. Accuracy: 93.56999969482422
Iteration: 2500. Loss: 0.0558248832821846. Accuracy: 95.69000244140625
Iteration: 3000. Loss: 0.09873700886964798. Accuracy: 95.58000183105469
Iteration: 3500. Loss: 0.08254366368055344. Accuracy: 96.8499984741211
Iteration: 4000. Loss: 0.03308983892202377. Accuracy: 97.08000183105469
Iteration: 4500. Loss: 0.049307383596897125. Accuracy: 97.08000183105469
Iteration: 5000. Loss: 0.08068697154521942. Accuracy: 96.69999694824219
Iteration: 5500. Loss: 0.12790139019489288. Accuracy: 97.30999755859375
Iteration: 6000. Loss: 0.007093742024153471. Accuracy: 97.66000366210938
Iteration: 6500. Loss: 0.011727366596460342. Accuracy: 97.63999938964844
Iteration: 7000. Loss: 0.008610538206994534. Accuracy: 98.06999969

In [14]:
def evaluate(model, val_iter):
    corrects, total, total_loss = 0, 0, 0
    model.eval()
    for images, labels in val_iter:
        if torch.cuda.is_available():
            images = Variable(images.view(-1, seq_dim, input_dim).cuda())
        else:
            images = Variable(images.view(-1 , seq_dim, input_dim)).to(device)
        labels = labels.cuda()
        logit = model(images).cuda()
        loss = F.cross_entropy(logit, labels, reduction = "sum")
        _, predicted = torch.max(logit.data, 1)
        total += labels.size(0)
        total_loss += loss.item()
        corrects += (predicted == labels).sum()

    avg_loss = total_loss / len(val_iter.dataset)
    avg_accuracy = corrects / total
    return avg_loss, avg_accuracy

In [15]:
test_loss, test_acc = evaluate(model,test_loader)
print("Test Loss: %5.2f | Test Accuracy: %5.2f" % (test_loss, test_acc))

Test Loss:  0.07 | Test Accuracy:  0.98


### **GRU 셀 구현 (과제)**

In [ ]:
# GRUCell

# Lstm과의 차이점에 유의

In [16]:
import torch
import torch.nn as nn
import torchvision.transforms as transforms
import torchvision.datasets as dataset
from torch.autograd import Variable
from torch.nn import Parameter
from torch import Tensor
import torch.nn.functional as F
from torch.utils.data import DataLoader
import math

device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
cuda = True if torch.cuda.is_available() else False

Tensor = torch.cuda.FloatTensor if cuda else torch.FloatTensor

torch.manual_seed(125)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(125)

In [17]:
mnist_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (1.0,))
])

In [18]:
from torchvision.datasets import MNIST
download_root = 'MNIST_DATASET/'

train_dataset = MNIST(download_root, transform=mnist_transform, train=True, download=True)
valid_dataset = MNIST(download_root, transform=mnist_transform, train=False, download=True)
test_dataset = MNIST(download_root, transform=mnist_transform, train=False, download=True)

In [19]:
batch_size = 64
train_loader = DataLoader(dataset=train_dataset,
                         batch_size=batch_size,
                         shuffle=True)
valid_loader = DataLoader(dataset=test_dataset,
                         batch_size=batch_size,
                         shuffle=True)
test_loader = DataLoader(dataset=test_dataset,
                         batch_size=batch_size,
                         shuffle=True)

In [20]:
batch_size = 100
n_iters = 6000
num_epochs = n_iters / (len(train_dataset) / batch_size)
num_epochs = int(num_epochs)

**과제:** 아래의 GRU 셀을 구현하세요. ### 부분을 채워주시면 됩니다.

In [21]:
class GRUCell(nn.Module):
    def __init__(self, input_size, hidden_size, bias=True):
        super(GRUCell, self).__init__()
        self.input_size = input_size
        self.hidden_size = hidden_size
        self.bias = bias
        self.x2h = nn.Linear(input_size, 3 * hidden_size, bias=bias)   # ← 3H
        self.h2h = nn.Linear(hidden_size, 3 * hidden_size, bias=bias)  # ← 3H
        self.reset_parameters()

    def reset_parameters(self):
        std = 1.0 / math.sqrt(self.hidden_size)
        for w in self.parameters():
            w.data.uniform_(-std, std)

    def forward(self, x, hidden):
        x = x.view(-1, x.size(1))

        gate_x = self.x2h(x)
        gate_h = self.h2h(hidden)

        gate_x = gate_x.squeeze()
        gate_h = gate_h.squeeze()

        i_r, i_i, i_n = gate_x.chunk(3, 1)
        h_r, h_i, h_n = gate_h.chunk(3, 1)

        resetgate = F.sigmoid(i_r + h_r)  # r_t
        inputgate = F.sigmoid(i_i + h_i)  # z_t
        newgate = F.tanh(i_n + resetgate * h_n)  # n_t = tanh(x_n + r ⊙ (W_hn h))

        # h_t = (1 - z) ⊙ n + z ⊙ h_{t-1}  ==  n + z ⊙ (h_{t-1} - n)
        hy = newgate + inputgate * (hidden - newgate)
        return hy

In [22]:
class GRUModel(nn.Module):
    def __init__(self, input_dim, hidden_dim, layer_dim, output_dim, bias=True):
        super(GRUModel, self).__init__()
        self.hidden_dim = hidden_dim
        self.layer_dim = layer_dim
        self.gru_cell = GRUCell(input_dim, hidden_dim, layer_dim)
        self.fc = nn.Linear(hidden_dim, output_dim)

    def forward(self, x):
        if torch.cuda.is_available():
            h0 = Variable(torch.zeros(self.layer_dim, x.size(0), self.hidden_dim).cuda())
        else:
            h0 = Variable(torch.zeros(self.layer_dim, x.size(0), self.hidden_dim))

        outs = []
        hn = h0[0,:,:]

        for seq in range(x.size(1)):
            hn = self.gru_cell(x[:,seq,:], hn)
            outs.append(hn)

        out = outs[-1].squeeze()
        out = self.fc(out)
        return out

In [23]:
input_dim = 28
hidden_dim = 128
layer_dim = 1
output_dim = 10

model = GRUModel(input_dim, hidden_dim, layer_dim, output_dim)

if torch.cuda.is_available():
    model.cuda()

criterion = nn.CrossEntropyLoss()
learning_rate = 0.1
optimizer = torch.optim.SGD(model.parameters(), lr=learning_rate)

In [24]:
seq_dim = 28
loss_list = []
iter = 0
for epoch in range(num_epochs):
    for i, (images, labels) in enumerate(train_loader):
        if torch.cuda.is_available():
            images = Variable(images.view(-1, seq_dim, input_dim).cuda())
            labels = Variable(labels.cuda())
        else:
            images = Variable(images.view(-1, seq_dim, input_dim))
            labels = Variable(labels)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)

        if torch.cuda.is_available():
            loss.cuda()

        loss.backward()
        optimizer.step()

        loss_list.append(loss.item())
        iter += 1

        if iter % 500 == 0:
            correct = 0
            total = 0
            for images, labels in valid_loader:
                if torch.cuda.is_available():
                    images = Variable(images.view(-1, seq_dim, input_dim).cuda())
                else:
                    images = Variable(images.view(-1 , seq_dim, input_dim))

                outputs = model(images)
                _, predicted = torch.max(outputs.data, 1)
                total += labels.size(0)

                if torch.cuda.is_available():
                    correct += (predicted.cpu() == labels.cpu()).sum()
                else:
                    correct += (predicted == labels).sum()

            accuracy = 100 * correct / total
            print('Iteration: {}. Loss: {}. Accuracy: {}'.format(iter, loss.item(), accuracy))

Iteration: 500. Loss: 1.6616928577423096. Accuracy: 43.59000015258789
Iteration: 1000. Loss: 0.8945668339729309. Accuracy: 76.19999694824219
Iteration: 1500. Loss: 0.29147759079933167. Accuracy: 89.7300033569336
Iteration: 2000. Loss: 0.23627927899360657. Accuracy: 93.51000213623047
Iteration: 2500. Loss: 0.03288726136088371. Accuracy: 95.05000305175781
Iteration: 3000. Loss: 0.030374974012374878. Accuracy: 95.81999969482422
Iteration: 3500. Loss: 0.16210567951202393. Accuracy: 96.33999633789062
Iteration: 4000. Loss: 0.19308766722679138. Accuracy: 96.19000244140625
Iteration: 4500. Loss: 0.051720067858695984. Accuracy: 97.0
Iteration: 5000. Loss: 0.13900163769721985. Accuracy: 97.26000213623047
Iteration: 5500. Loss: 0.08090294152498245. Accuracy: 97.62000274658203
Iteration: 6000. Loss: 0.10488356649875641. Accuracy: 97.69000244140625
Iteration: 6500. Loss: 0.07984025031328201. Accuracy: 97.80000305175781
Iteration: 7000. Loss: 0.10250380635261536. Accuracy: 97.55999755859375
Iterati

In [25]:
def evaluate(model, val_iter):
    corrects, total, total_loss = 0, 0, 0
    model.eval()
    for images, labels in val_iter:
        if torch.cuda.is_available():
            images = Variable(images.view(-1, seq_dim, input_dim).cuda())
        else:
            images = Variable(images.view(-1 , seq_dim, input_dim)).to(device)
        labels = labels.cuda()
        logit = model(images).cuda()
        loss = F.cross_entropy(logit, labels, reduction = "sum")
        _, predicted = torch.max(logit.data, 1)
        total += labels.size(0)
        total_loss += loss.item()
        corrects += (predicted == labels).sum()

    avg_loss = total_loss / len(val_iter.dataset)
    avg_accuracy = corrects / total
    return avg_loss, avg_accuracy

In [26]:
test_loss, test_acc = evaluate(model,test_loader)
print("Test Loss: %5.2f | Test Accuracy: %5.2f" % (test_loss, test_acc))

Test Loss:  0.07 | Test Accuracy:  0.98


## 과제 2

1. **LSTM 기반 Seq2Seq 모델에서 디코딩할 때 사용하는 Beam Search 동작 방식에 대해서 설명해주세요.**

LSTM 기반 Seq2Seq 모델에서 디코딩할 때는 좌에서 우로 향하는 Beam Search 디코더를 사용합니다. \

이때 Beam Search는 완성되지 않은 B개의 부분 가설을 유지하며, 이후 각 시점에서 현재 유지하고 있는 부분 가설 각각에 대해 가능한 모든 단어를 추가해 확률이 높은 상위 B개의 가설만 유지합니다. 번역 가설 내에서 <EOS> 토큰이 나오면 해당 가설은 완성된 번역으로 간주하여 이동시킵니다.

예를 들어 beam size가 2라고 가정하면,
첫 번째 단어를 예측했을 때 확률이 가장 높은 두 단어 "I"(0.6)와 "He"(0.4)가 선택됩니다. 이때의 부분 문장 후보는 I와 He입니다.
다음 시점에서는 각 부분 문장에 가능한 모든 단어를 붙여 새로운 후보를 만듭니다. 예를 들어 단어 후보가 "am", "is", "like"라면, 가능한 모든 단어로 확장한
- I am (0.6×0.5=0.30)
- I is (0.6×0.2=0.12)
- I like (0.6×0.3=0.18)
- He am (0.4×0.4=0.16)
- He is (0.4×0.4=0.16)
- He like (0.4×0.2=0.08) \
의 6개 단어가 새로운 후보가 됩니다. 이중에서 확률이 가장 높은 상위 2개 후보만 남깁니다. 위의 예시로는 I am (0.30), I like (0.18)가 남습니다. 이 과정을 반복하다가 <EOS> 토큰이 나오면 해당 문장을 완성된 번역으로 간주해 최종 후보 리스트에 넣는 방식입니다.















**2. Seq2Seq with LSTM 모델은 Attention이 없던 시절 제안된 구조입니다. 기본 Seq2Seq 모델의 한계와 이후 Attention 메커니즘이 이 한계를 어떻게 보완했는지 설명해주세요.**

기본 Seq2Seq (LSTM 기반) 모델의 한계

1. 고정된 크기의 벡터 제약 \
: 초기 딥러닝 모델(DNN)은 입력과 출력이 고정된 크기의 벡터로 인코딩되는 문제에 주로 적용 가능했으며, 길이가 가변적인 Sequence to Sequence 문제에서는 사용하기 어려웠습니다. 논문에서 제안한 Seq2Seq 모델은 이 문제를 해결하고자 했지만, 여전히 입력 시퀀스 전체를 하나의 고정된 크기의 벡터 v로 압축하는 방식이었습니다.

2. 정보 손실 및 장기 의존성 문제 \
: 인코더 LSTM은 입력 시퀀스로부터 고정된 크기의 벡터 v를 얻고, 이 v가 디코더 LSTM의 첫 은닉층 입력값이 됩니다. 즉 LSTM은 기본적으로 입력 전체를 마지막 hidden state 하나에 압축하기 때문에, 문장이 길어질수록 앞부분의 정보가 뒤로 갈수록 희미해져 결국 정보 손실이 발생합니다. LSTM이 RNN보다는 장기 의존성 문제를 많이 완화했지만, 여전히 입력 전체를 마지막 hidden state 하나에 담아야 한다는 구조적 한계가 있었습니다.

3. Attention 메커니즘의 한계 보완 \
: Attention은 디코더가 단어를 생성할 때마다 인코더의 전체 hidden state들을 다시 볼 수 있게 만듭니다. 매 시점마다 각 hidden state가 얼마나 중요한지 가중치(attention weight)를 계산하고, 이렇게 계산된 가중치를 이용해 인코더 출력들의 가중합(context vector)을 만들고, 이 벡터를 디코더에 함께 넣어줍니다. 이를 통해 고정된 하나의 컨텍스트 벡터에 모든 정보를 압축할 필요 없이, 필요한 시점에 필요한 정보를 직접 가져올 수 있게 되어 긴 시퀀스에서도 정보 손실 없이 효율적으로 장기 의존성 문제를 해결할 수 있게 됩니다.
즉, Attention은 디코더가 매 시점마다 인코더 전체 출력 중 중요한 부분을 집중해서 활용하도록 만들어, 정보 병목과 장기 의존성 문제를 해결합니다.